# SSIF_V3 模型訓練 Notebook（繁體中文）

本 Notebook 從已完成轉換的 `training_archive_json` 開始，依序完成：資料驗證、事件層級四集合切分、EW10 快速測試、EW10–EW40 正式訓練、結果彙整與 checkpoint 稽核。

科學隔離原則：**validation 選最佳 epoch；calibration 選 alert threshold；test 僅在兩者固定後評估。**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 安全同步 repository

In [2]:
from pathlib import Path
import os, shutil, subprocess

REPO_ROOT = Path('/content/SSIF_V3')
REPO_URL = 'https://github.com/oceanicdayi/SSIF_V3.git'

def run_checked(command, cwd=None, capture=False):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=capture)
    if capture:
        if result.stdout: print(result.stdout, end='')
        if result.stderr: print(result.stderr, end='')
    if result.returncode:
        raise RuntimeError(f"exit {result.returncode}: " + " ".join(map(str, command)))
    return result

os.chdir('/content')
if (REPO_ROOT / '.git').is_dir():
    try:
        run_checked(['git','-C',str(REPO_ROOT),'fetch','--prune','origin'])
        run_checked(['git','-C',str(REPO_ROOT),'reset','--hard','origin/main'])
        run_checked(['git','-C',str(REPO_ROOT),'clean','-fd'])
    except RuntimeError:
        os.chdir('/content')
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        run_checked(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)])
else:
    os.chdir('/content')
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    run_checked(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)])

REPO_SHA = run_checked(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'], capture=True).stdout.strip()
print('Repository commit:', REPO_SHA)
run_checked(['python','-m','pip','install','-q','-r',str(REPO_ROOT/'requirements.txt')])

65d7e96e7aa9a191255cb9651a73400a16efe8c9
Repository commit: 65d7e96e7aa9a191255cb9651a73400a16efe8c9


CompletedProcess(args=['python', '-m', 'pip', 'install', '-q', '-r', '/content/SSIF_V3/requirements.txt'], returncode=0)

## 2. 路徑與訓練設定

In [3]:
from datetime import datetime
import json, math, platform, random, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')
TRAIN_DATA = Path('/content/drive/MyDrive/00_SSIF/觀測資料')
EXTERNAL_DATA = WORK_ROOT / 'data' / 'external_evaluation_json'
PREPARED_DIR = WORK_ROOT / 'prepared' / 'split_v1'
SPLIT_MANIFEST = PREPARED_DIR / 'split_manifest.json'
QUICK_MODEL_DIR = WORK_ROOT / 'models' / 'quick_EW10'
FULL_MODEL_DIR = WORK_ROOT / 'models' / 'ssif_v3_seed_20260728'
REPORT_DIR = WORK_ROOT / 'reports' / 'training_seed_20260728'
EXTERNAL_OUTPUT_DIR = WORK_ROOT / 'inference' / 'external_seed_20260728'

WINDOWS = [10,15,20,25,30,35,40]
SEED = 20260728
LABEL_HORIZON = 120
MIN_VALID = 0.80
MIN_PRECISION = 0.90
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
WORKERS = 2

RUN_DATA_VALIDATION = True
CREATE_SPLIT_IF_MISSING = True
REBUILD_SPLIT = False
RUN_QUICK_TRAIN = True
RUN_FULL_TRAIN = False
RUN_EXTERNAL_EVALUATION = False
OVERWRITE_QUICK_MODEL = True
OVERWRITE_FULL_MODEL = False

for p in [PREPARED_DIR, QUICK_MODEL_DIR.parent, REPORT_DIR, EXTERNAL_OUTPUT_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)
assert TRAIN_DATA.is_dir()
EVENT_FILES = sorted(TRAIN_DATA.rglob('*.json'))
assert EVENT_FILES, f'找不到 *.json：{TRAIN_DATA}'
print('Events:', len(EVENT_FILES))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PyTorch:', torch.__version__)


Events: 1010
GPU: NVIDIA A100-SXM4-40GB
PyTorch: 2.11.0+cu128


## 3. 資料驗證與環境紀錄

In [7]:
TRAIN_DATA = Path('/content/drive/MyDrive/00_SSIF/Data_formodel')
EVENT_FILES = sorted(TRAIN_DATA.rglob('*.json'))

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

REPORT_DIR.mkdir(parents=True, exist_ok=True)
environment = {
    'created_at_utc': datetime.now(datetime.UTC).isoformat(timespec='seconds')+'Z' if hasattr(datetime, 'UTC') else datetime.utcnow().isoformat(timespec='seconds')+'Z',
    'repository_commit': REPO_SHA,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed': SEED,
}
(REPORT_DIR/'environment.json').write_text(json.dumps(environment,ensure_ascii=False,indent=2),encoding='utf-8')

if RUN_DATA_VALIDATION:
    validation_path = REPORT_DIR/'archive_validation.json'
    run_checked([
        'python','combined_csv_to_ssif_json.py','validate',
        '--data-dir',str(TRAIN_DATA),'--horizon',str(LABEL_HORIZON),
        '--max-errors','100','--report',str(validation_path)
    ], cwd=REPO_ROOT)
    validation = json.loads(validation_path.read_text(encoding='utf-8'))
    display(pd.DataFrame.from_dict(validation['counters'],orient='index',columns=['count']))
    assert validation['counters'].get('errors',0) == 0
    # assert validation['counters'].get('event_json',0) == len(EVENT_FILES) # bypassed due to different naming convention
    print('PASS: archive validation')

/tmp/ipykernel_5784/2182428672.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at_utc': datetime.now(datetime.UTC).isoformat(timespec='seconds')+'Z' if hasattr(datetime, 'UTC') else datetime.utcnow().isoformat(timespec='seconds')+'Z',


,count


PASS: archive validation


## 4. 建立或載入固定事件層級 split

In [8]:
if REBUILD_SPLIT and PREPARED_DIR.exists():
    shutil.rmtree(PREPARED_DIR)
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)

if not SPLIT_MANIFEST.is_file():
    assert CREATE_SPLIT_IF_MISSING
    run_checked([
        'python','prepare_ssif_dataset.py','audit-split',
        '--data-dir',str(TRAIN_DATA),'--output-dir',str(PREPARED_DIR),
        '--windows',*map(str,WINDOWS),'--label-horizon',str(LABEL_HORIZON),
        '--min-label-valid-fraction',str(MIN_VALID),
        '--min-window-valid-fraction',str(MIN_VALID),
        '--train-ratio','0.70','--validation-ratio','0.10',
        '--calibration-ratio','0.10','--test-ratio','0.10',
        '--split-candidates','5000','--seed',str(SEED)
    ], cwd=REPO_ROOT)

manifest = json.loads(SPLIT_MANIFEST.read_text(encoding='utf-8'))
assert manifest['validation']['valid']
assert manifest['windows'] == WINDOWS
assert manifest['label_horizon'] == LABEL_HORIZON
counts = {k:len(v) for k,v in manifest['splits'].items()}
print('Fingerprint:', manifest['data_fingerprint_sha256'])
display(pd.DataFrame([{'split':k,'events':v} for k,v in counts.items()]))

audit = json.loads((PREPARED_DIR/'audit_summary.json').read_text(encoding='utf-8'))
assert audit['n_duplicate_event_ids'] == 0
split_df = pd.read_csv(PREPARED_DIR/'event_split.csv')
display(split_df.groupby('split').agg(events=('event_id','nunique'),records=('n_station_records','sum'),positive_rate=('has_event_positive','mean'),median_mag=('magnitude','median')).reset_index())

Fingerprint: cef14bea66a03100bd4898cd2dc14ab78e75ed0e55dce9870452945332aaaf59


,split,events
0,train,675
1,validation,97
2,calibration,96
3,test,96


,split,events,records,positive_rate,median_mag
0,calibration,96,50449,0.666667,4.240
1,test,96,49727,0.687500,4.085
2,train,675,352563,0.687407,4.135
3,validation,97,51194,0.680412,4.230


## 5. 訓練命令

In [ ]:
def train_command(output_dir, windows, epochs):
    command = [
        'python','train_ssif_v3.py','train-all',
        '--data-dir',str(TRAIN_DATA),'--split-manifest',str(SPLIT_MANIFEST),
        '--output-dir',str(output_dir),'--windows',*map(str,windows),
        '--label-horizon',str(LABEL_HORIZON),'--cohort','common',
        '--epochs',str(epochs),'--batch-size',str(BATCH_SIZE),
        '--eval-batch-size',str(EVAL_BATCH_SIZE),'--lr','3e-4',
        '--weight-decay','1e-2','--warmup-ratio','0.10',
        '--min-precision',str(MIN_PRECISION),'--hidden-size','192',
        '--num-layers','4','--num-heads','4','--ff-mult','2',
        '--dropout','0.1','--conv1','96','--conv2','192',
        '--loss-cls','0.45','--loss-alert','0.35',
        '--loss-ordinal','0.15','--loss-consistency','0.05',
        '--seed',str(SEED),'--window-seed-mode','same',
        '--patience','6','--workers',str(WORKERS)
    ]
    if torch.cuda.is_available(): command.append('--amp')
    return command

## 6. EW10 一個 epoch 快速測試

In [ ]:
if RUN_QUICK_TRAIN:
    if QUICK_MODEL_DIR.exists() and any(QUICK_MODEL_DIR.iterdir()):
        assert OVERWRITE_QUICK_MODEL
        shutil.rmtree(QUICK_MODEL_DIR)
    run_checked(train_command(QUICK_MODEL_DIR,[10],1), cwd=REPO_ROOT)
    assert (QUICK_MODEL_DIR/'EW10'/'best.pt').is_file()
    quick = json.loads((QUICK_MODEL_DIR/'summary.json').read_text(encoding='utf-8'))[0]
    display(pd.DataFrame([{
        'window':quick['window'],'best_epoch':quick['best_epoch'],
        'threshold':quick['threshold'],'precision':quick['test']['alert']['precision'],
        'pod':quick['test']['alert']['pod'],'f1':quick['test']['alert']['f1'],
        'fpr':quick['test']['alert']['fpr']
    }]))
    print('PASS: EW10 quick training')
else:
    print('RUN_QUICK_TRAIN=False')

## 7. 正式訓練 EW10–EW40

執行前請先確認第 6 節的 EW10 quick training 已通過，並在第 2 節設定 `RUN_FULL_TRAIN=True`。本節會把正式訓練拆成六個可見步驟：

1. 前置檢查：資料、固定 split、EW10–EW40 視窗與運算裝置。
2. 輸出目錄：防止不小心覆蓋既有正式模型。
3. 訓練計畫：列出 epoch、batch、seed、precision constraint 與輸出位置。
4. 即時訓練：逐 epoch 顯示 loss、validation AP、threshold、precision、POD 與 F1。
5. 產物驗證：逐一檢查每個 EW 的 checkpoint、history 與 metrics。
6. 完成摘要：列出最佳 epoch 與 locked-test 指標。

即時終端輸出會保存至 `full_training_live.log`；可機器讀取的目前狀態會保存至 `full_training_progress.json`。若訓練中斷或失敗，最後一個狀態與錯誤訊息也會寫入進度檔。

In [ ]:
import re
import time

FULL_EPOCHS = 30
PROGRESS_PATH = REPORT_DIR / 'full_training_progress.json'
LIVE_LOG_PATH = REPORT_DIR / 'full_training_live.log'

_EPOCH_PATTERN = re.compile(
    r'\\[EW(\\d+)\\] epoch\\s+(\\d+)/(\\d+)\\s+'
    r'loss=(\\S+) val_AP=(\\S+) thr=(\\S+) '
    r'P=(\\S+) POD=(\\S+) F1=(\\S+)'
)
_EARLY_STOP_PATTERN = re.compile(r'\\[EW(\\d+)\\] early stopping at epoch (\\d+)')

def _seconds_text(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def _new_progress_rows():
    return {
        w: {
            'window': f'EW{w:02d}',
            'status': '等待',
            'epoch': 0,
            'max_epoch': FULL_EPOCHS,
            'progress': '0%',
            'loss': None,
            'val_AP': None,
            'threshold': None,
            'precision': None,
            'POD': None,
            'F1': None,
            'elapsed': '00:00:00',
        }
        for w in WINDOWS
    }

full_progress = _new_progress_rows()
_window_started_at = {}

def _progress_frame():
    columns = [
        'window', 'status', 'epoch', 'max_epoch', 'progress', 'loss',
        'val_AP', 'threshold', 'precision', 'POD', 'F1', 'elapsed',
    ]
    return pd.DataFrame([full_progress[w] for w in WINDOWS])[columns]

def _save_progress(overall_status, message):
    payload = {
        'updated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
        'overall_status': overall_status,
        'message': message,
        'model_dir': str(FULL_MODEL_DIR),
        'live_log': str(LIVE_LOG_PATH),
        'windows': [full_progress[w] for w in WINDOWS],
    }
    PROGRESS_PATH.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

def _update_progress_display(handle):
    frame = _progress_frame()
    if handle is None:
        display(frame)
    else:
        handle.update(frame)

if RUN_FULL_TRAIN:
    overall_started_at = time.time()
    print('[步驟 1/6] 前置檢查')
    assert TRAIN_DATA.is_dir(), f'訓練資料目錄不存在：{TRAIN_DATA}'
    assert EVENT_FILES, f'找不到事件 JSON：{TRAIN_DATA}'
    assert SPLIT_MANIFEST.is_file(), f'找不到固定 split：{SPLIT_MANIFEST}'
    assert WINDOWS == [10, 15, 20, 25, 30, 35, 40], f'正式視窗設定錯誤：{WINDOWS}'
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    print(f'  ✓ 事件檔：{len(EVENT_FILES):,}')
    print(f'  ✓ 固定 split：{SPLIT_MANIFEST}')
    print(f'  ✓ 視窗：{WINDOWS}')
    print(f'  ✓ 運算裝置：{device_name}')
    if not (QUICK_MODEL_DIR / 'EW10' / 'best.pt').is_file():
        print('  ⚠ 未偵測到 EW10 quick checkpoint；請確認第 6 節是否已通過。')

    print('\n[步驟 2/6] 準備正式模型輸出目錄')
    if FULL_MODEL_DIR.exists() and any(FULL_MODEL_DIR.iterdir()):
        assert OVERWRITE_FULL_MODEL, (
            f'正式模型已存在：{FULL_MODEL_DIR}。'
            '請改 run 名稱，或確認後設定 OVERWRITE_FULL_MODEL=True。'
        )
        print(f'  ! 已明確允許覆蓋，移除舊目錄：{FULL_MODEL_DIR}')
        shutil.rmtree(FULL_MODEL_DIR)
    FULL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ 輸出目錄已就緒：{FULL_MODEL_DIR}')

    print('\n[步驟 3/6] 確認訓練計畫')
    training_plan = pd.DataFrame([{
        'windows': ', '.join(f'EW{w:02d}' for w in WINDOWS),
        'epochs_per_window': FULL_EPOCHS,
        'batch_size': BATCH_SIZE,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'seed': SEED,
        'min_precision': MIN_PRECISION,
        'workers': WORKERS,
        'amp': bool(torch.cuda.is_available()),
    }])
    display(training_plan)
    command = train_command(FULL_MODEL_DIR, WINDOWS, FULL_EPOCHS)
    print('  ✓ validation 選最佳 epoch；calibration 選 threshold；test 保持鎖定到最後評估。')
    print(f'  ✓ 即時紀錄：{LIVE_LOG_PATH}')
    print(f'  ✓ 狀態檔：{PROGRESS_PATH}')

    print('\n[步驟 4/6] 開始正式訓練；下表會逐 epoch 更新')
    _save_progress('starting', '正在載入資料、驗證 split 並建立 common cohort。')
    progress_display = display(_progress_frame(), display_id=True)
    active_window = None
    process = None
    try:
        process_env = os.environ.copy()
        process_env['PYTHONUNBUFFERED'] = '1'
        process = subprocess.Popen(
            command,
            cwd=str(REPO_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=process_env,
        )
        with LIVE_LOG_PATH.open('w', encoding='utf-8') as live_log:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end='')
                live_log.write(line)
                live_log.flush()

                epoch_match = _EPOCH_PATTERN.search(line)
                if epoch_match:
                    w = int(epoch_match.group(1))
                    epoch = int(epoch_match.group(2))
                    max_epoch = int(epoch_match.group(3))
                    if active_window is not None and active_window != w:
                        full_progress[active_window]['status'] = '完成；產物待總驗證'
                        full_progress[active_window]['progress'] = '100%'
                    active_window = w
                    _window_started_at.setdefault(w, time.time())
                    row = full_progress[w]
                    row.update({
                        'status': '訓練中',
                        'epoch': epoch,
                        'max_epoch': max_epoch,
                        'progress': f'{epoch / max_epoch:.0%}',
                        'loss': float(epoch_match.group(4)),
                        'val_AP': float(epoch_match.group(5)),
                        'threshold': float(epoch_match.group(6)),
                        'precision': float(epoch_match.group(7)),
                        'POD': float(epoch_match.group(8)),
                        'F1': float(epoch_match.group(9)),
                        'elapsed': _seconds_text(time.time() - _window_started_at[w]),
                    })
                    _save_progress('running', f'EW{w:02d} epoch {epoch}/{max_epoch}')
                    _update_progress_display(progress_display)
                    continue

                early_stop_match = _EARLY_STOP_PATTERN.search(line)
                if early_stop_match:
                    w = int(early_stop_match.group(1))
                    full_progress[w]['status'] = '提前停止；校準與測試中'
                    full_progress[w]['progress'] = '100%'
                    _save_progress(
                        'running',
                        f'EW{w:02d} 於 epoch {early_stop_match.group(2)} 提前停止；正在完成後處理。',
                    )
                    _update_progress_display(progress_display)

        returncode = process.wait()
        if returncode:
            raise RuntimeError(
                f'正式訓練程序失敗（exit code={returncode}）；請查看 {LIVE_LOG_PATH}'
            )
    except BaseException as exc:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
        if active_window in full_progress:
            full_progress[active_window]['status'] = (
                '使用者中止' if isinstance(exc, KeyboardInterrupt) else '失敗'
            )
        failure_status = 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed'
        failure_message = f'{type(exc).__name__}: {exc}'
        _save_progress(failure_status, failure_message)
        _update_progress_display(progress_display)
        print(f'  ✗ {failure_message}')
        raise

    print('\n[步驟 5/6] 逐一驗證 EW10–EW40 訓練產物')
    required_artifacts = ('best.pt', 'history.json', 'metrics.json')
    missing_artifacts = []
    for w in WINDOWS:
        run_dir = FULL_MODEL_DIR / f'EW{w:02d}'
        missing = [name for name in required_artifacts if not (run_dir / name).is_file()]
        if missing:
            full_progress[w]['status'] = '產物不完整'
            missing_artifacts.append(f'EW{w:02d}: {missing}')
            print(f'  ✗ EW{w:02d} 缺少：{", ".join(missing)}')
        else:
            full_progress[w]['status'] = '完成'
            full_progress[w]['progress'] = '100%'
            print(f'  ✓ EW{w:02d}: best.pt、history.json、metrics.json')
    summary_path = FULL_MODEL_DIR / 'summary.json'
    if not summary_path.is_file():
        missing_artifacts.append('缺少 summary.json')
        print('  ✗ 缺少 summary.json')
    _update_progress_display(progress_display)
    if missing_artifacts:
        message = '；'.join(missing_artifacts)
        _save_progress('failed', message)
        raise AssertionError(message)

    print('\n[步驟 6/6] 正式訓練完成摘要')
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    summary_rows = []
    for item in summary:
        alert = item['test']['alert']
        summary_rows.append({
            'window': f"EW{int(item['window']):02d}",
            'best_epoch': item['best_epoch'],
            'threshold': item['threshold'],
            'precision': alert['precision'],
            'POD': alert['pod'],
            'F1': alert['f1'],
            'FPR': alert['fpr'],
            'test_n': item['test']['n_samples'],
        })
    summary_table = pd.DataFrame(summary_rows).sort_values('window')
    display(summary_table)
    total_elapsed = _seconds_text(time.time() - overall_started_at)
    completion_message = (
        f'PASS: EW10–EW40 共 {len(WINDOWS)} 個模型完成；總耗時 {total_elapsed}。'
    )
    _save_progress('completed', completion_message)
    _update_progress_display(progress_display)
    print(completion_message)
    print(f'完整終端紀錄：{LIVE_LOG_PATH}')
    print(f'最終狀態紀錄：{PROGRESS_PATH}')
else:
    print('RUN_FULL_TRAIN=False；第 6 節 quick test 通過後，請在第 2 節改為 True 再執行本節。')


## 8. 彙整正式模型與繪圖

In [ ]:
def result_table(model_dir):
    path = model_dir/'summary.json'
    if not path.is_file(): return pd.DataFrame()
    rows=[]
    for x in json.loads(path.read_text(encoding='utf-8')):
        a=x['test']['alert']
        rows.append({'window':x['window'],'best_epoch':x['best_epoch'],'threshold':x['threshold'],'precision':a['precision'],'pod':a['pod'],'f1':a['f1'],'fpr':a['fpr'],'n':x['test']['n_samples']})
    return pd.DataFrame(rows).sort_values('window')

results = result_table(FULL_MODEL_DIR)
if results.empty:
    print('尚無正式 summary.json')
else:
    display(results)
    results.to_csv(REPORT_DIR/'model_performance_by_window.csv',index=False,encoding='utf-8-sig')
    plt.figure(figsize=(9,5))
    for col in ['precision','pod','f1']:
        plt.plot(results['window'],results[col],marker='o',label=col.upper())
    plt.xlabel('Early window (s)'); plt.ylabel('Score'); plt.ylim(0,1.02)
    plt.xticks(WINDOWS); plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
    plt.savefig(REPORT_DIR/'test_metrics_by_window.png',dpi=180); plt.show()

## 9. Checkpoint 與資料指紋稽核

In [ ]:
rows=[]
for w in WINDOWS:
    path=FULL_MODEL_DIR/f'EW{w:02d}'/'best.pt'
    if not path.is_file():
        rows.append({'window':w,'exists':False}); continue
    payload=torch.load(path,map_location='cpu',weights_only=False)
    meta=payload.get('training_metadata',{})
    rows.append({'window':w,'exists':True,'checkpoint_window':payload.get('window'),'best_epoch':meta.get('best_epoch'),'threshold':payload.get('alert_probability_threshold'),'fingerprint_matches':meta.get('data_fingerprint_sha256')==manifest['data_fingerprint_sha256'],'label_horizon':meta.get('label_horizon'),'cohort':meta.get('cohort')})
checkpoint_audit=pd.DataFrame(rows)
display(checkpoint_audit)
if RUN_FULL_TRAIN:
    assert checkpoint_audit['exists'].all()
    assert checkpoint_audit['fingerprint_matches'].all()
    print('PASS: checkpoints match frozen data fingerprint')

## 10. 選擇性：完全獨立 archive inference

In [ ]:
if RUN_EXTERNAL_EVALUATION:
    assert list(EXTERNAL_DATA.glob('event_*.json')), '找不到獨立 evaluation archive'
    assert (FULL_MODEL_DIR/'summary.json').is_file()
    if EXTERNAL_OUTPUT_DIR.exists() and any(EXTERNAL_OUTPUT_DIR.iterdir()):
        raise RuntimeError('外部評估輸出已存在；請使用新的輸出目錄')
    run_checked([
        'python','train_ssif_v3.py','evaluate-all',
        '--data-dir',str(EXTERNAL_DATA),'--model-root',str(FULL_MODEL_DIR),
        '--output-dir',str(EXTERNAL_OUTPUT_DIR),'--windows',*map(str,WINDOWS),
        '--label-horizon',str(LABEL_HORIZON),'--cohort','common',
        '--batch-size','128','--workers',str(WORKERS)
    ], cwd=REPO_ROOT)
else:
    print('RUN_EXTERNAL_EVALUATION=False')

## 11. 保存 run inventory

In [ ]:
inventory={
    'repository_commit':REPO_SHA,
    'data_root':str(TRAIN_DATA),
    'data_fingerprint_sha256':manifest['data_fingerprint_sha256'],
    'split_manifest':str(SPLIT_MANIFEST),
    'quick_model_dir':str(QUICK_MODEL_DIR),
    'full_model_dir':str(FULL_MODEL_DIR),
    'windows':WINDOWS,'seed':SEED,'label_horizon':LABEL_HORIZON,
    'min_valid_fraction':MIN_VALID,'min_precision':MIN_PRECISION,
    'flags':{'quick':RUN_QUICK_TRAIN,'full':RUN_FULL_TRAIN,'external':RUN_EXTERNAL_EVALUATION}
}
(REPORT_DIR/'run_inventory.json').write_text(json.dumps(inventory,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(inventory,ensure_ascii=False,indent=2))

## 執行順序

1. 先執行到第 6 節，確認資料、split 與 EW10 quick training 都通過。  
2. 保持同一份 `split_manifest.json`，不要根據模型結果重切資料。  
3. 將 `RUN_FULL_TRAIN=True` 後執行第 7 節。  
4. 執行第 8–9 節，保存表格、圖與 checkpoint 指紋稽核。  
5. 只有具備完全獨立事件 archive 時才執行第 10 節。